In [1]:
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd

# Point to project root
sys.path.append(os.path.abspath('..'))

from src.preprocessing import split_and_scale_timeseries, create_sliding_windows

# Load clean CSV from Phase 2
csv_path = Path('../data/ssl_pretraining/lcl_2013_multivariate.csv')
df = pd.read_csv(csv_path, index_col=0, parse_dates=True)
print(f"Loaded matrix: {df.shape}")

Loaded matrix: (17520, 25)


In [2]:
# 1. Chronological Split & Leakage-Free Scaling
train_scaled, val_scaled, test_scaled, scaler = split_and_scale_timeseries(df, train_ratio=0.70, val_ratio=0.15)

# 2. Rolling Sliding Windows (48 steps = 24-hour diurnal cycle)
WINDOW_SIZE = 48
STEP_SIZE = 1

X_train = create_sliding_windows(train_scaled, window_size=WINDOW_SIZE, step_size=STEP_SIZE)
X_val = create_sliding_windows(val_scaled, window_size=WINDOW_SIZE, step_size=STEP_SIZE)
X_test = create_sliding_windows(test_scaled, window_size=WINDOW_SIZE, step_size=STEP_SIZE)

print("--- Output Tensors ---")
print(f"X_train: {X_train.shape} -> (samples, window_size, features)")
print(f"X_val:   {X_val.shape}")
print(f"X_test:  {X_test.shape}")

# 3. Save model-ready tensors to disk
output_file = Path('../data/ssl_pretraining/lcl_windows.npz')
np.savez_compressed(output_file, X_train=X_train, X_val=X_val, X_test=X_test)
print(f"Successfully saved to: {output_file}")

--- Output Tensors ---
X_train: (12217, 48, 25) -> (samples, window_size, features)
X_val:   (2581, 48, 25)
X_test:  (2581, 48, 25)
Successfully saved to: ..\data\ssl_pretraining\lcl_windows.npz
